<a href="https://colab.research.google.com/github/Subroy1/MSAI_AllPracticeModules/blob/main/Module%2013/Assignment13_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Required Assignment 13.3: End-to-End Classification Problem

**Expected Time = 90 minutes**

**Total Points = 85**

This example leads you through an end-to-end analysis of a classification algorithm using `LogisticRegression`. You will perform some brief exploratory data analysis (EDA). Then, you will construct a feature engineering, selection, and model pipeline. Finally, you will explore the mistakes your models make and compare different classification metrics.

### Index

- [Problem 1](#-Problem-1)
- [Problem 2](#-Problem-2)
- [Problem 3](#-Problem-3)
- [Problem 4](#-Problem-4)
- [Problem 5](#-Problem-5)
- [Problem 6](#-Problem-6)
- [Problem 7](#-Problem-7)
- [Problem 8](#-Problem-8)
- [Problem 9](#-Problem-9)
- [Problem 10](#-Problem-10)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.metrics import roc_curve, auc

### The Data

This data is originally from the IBM and contains information on a telecommunications company customer subscriptions.  Your task is to predict the customers who will Churn.  The data is loaded, displayed, and split below.

In [ ]:
df = pd.read_csv('/content/sample_data/wa_churn.csv')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan).astype('float')

In [ ]:
df = df.dropna()

In [ ]:
df['Churn'].map({'Yes': 1, 'No': 0}).sum()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(['Churn', 'customerID'], axis = 1), df['Churn'], random_state = 442, stratify = df['Churn'])# stratify is used to handle the imbalance in the data

[Back to top](#-Index)

### Problem 1

### `make_column_selector`

**5 Points**

To begin, you may want to incorporate many of the categorical features here.  Rather than writing a list of names, you can use the `make_column_selector` to select features by datatype.  For example:

```python
make_column_selector(dtype_include=object)
```

will select all columns with `object` datatype.  This selector will replace the list of column names in the `make_column_transformer`.  

Create a selector object to select the columns with `object` datatype below.  Assign this to `selector`.

In [ ]:
### GRADED
selector = ''
# YOUR CODE HERE
selector = make_column_selector(dtype_include = object)

# Answer check
selector


In [ ]:
selector(X_train)

[Back to top](#-Index)

### Problem 2

#### Column Transformer

**5 Points**

Use the `make_column_transformer` function on the the columns selected by `selector`. To these columns, apply the `OneHotEncoder` with `drop = first`. To the `remainder` columns, apply `StandardScaler()`

Assign the result to `transformer` below.



In [ ]:
### GRADED
transformer = ''
# YOUR CODE HERE
ohe = OneHotEncoder(drop='first', handle_unknown= 'ignore')
sc = StandardScaler()
transformer= make_column_transformer((ohe, selector(X_train)), remainder=sc)

# Answer check
transformer

[Back to top](#-Index)

### Problem 3

#### Feature Extractor

**5 Points**

Just as in our earlier assignment you can use `LogisticRegression` with `l1` penalty to select features for the model.

Below, create a `SelectFromModel` object that uses a `LogisticRegression` estimator with `penalty = 'l1'`, solver of `liblinear` and `random_state = 42`.  Assign your transformer as `extractor` below.

In [ ]:
### GRADED
extractor = ''
# YOUR CODE HERE
extractor = SelectFromModel(LogisticRegression(penalty='l1',  solver='liblinear', random_state=42))
# Answer check
extractor

[Back to top](#-Index)

### Problem 4

#### Pipeline with Estimator

**10 Points**

Now, build a pipeline `lgr_pipe` with named steps `transformer`, `selector`, and `lgr` that implement the column transformer, feature selector from above and a `LogisticRegression` estimator with `random_state = 42` and `max_iter = 1000`.  

Fit the pipeline on the training data and determine the score on the test data.  

Finally, use the function `score` to calculate the accuracy as a float to `pipe_1_acc` below.

In [ ]:
### GRADED
lgr_pipe = ''
pipe_1_acc = ''
# YOUR CODE HERE
lgr_pipe = Pipeline(steps = [
  ("transformer",transformer ),
  ("selector", extractor),
  ("lgr",LogisticRegression(random_state=42, max_iter=1000))
])
lgr_pipe.fit(X_train, y_train)
y_pred= lgr_pipe.predict(X_test)
pipe_1_acc = lgr_pipe.score(X_test, y_test)

# Answer check
print(pipe_1_acc)
lgr_pipe


[Back to top](#-Index)

### Problem 5

#### Comparison to Baseline

**10 Points**

Use the `value_counts` method to determine the baseline score by choosing the majority class as your predictions.  Did your pipeline outperform the baseline model?  Answer `yes` or `no` as a string to `ans5` below.

In [ ]:
pd.DataFrame(y_pred)[0].map({'Yes':1, 'No':0}).value_counts()

In [ ]:
pipe_1_acc

In [ ]:
pd.DataFrame(lgr_pipe.predict(X_test)).value_counts()

In [ ]:
y_test.value_counts()

In [ ]:
### GRADED
ans5 = ''
# YOUR CODE HERE
Pn = 1291/(1291+467) # majority class
ans5 = (pipe_1_acc>Pn)

if (ans5==True):
  ans5= "yes"
else:
  ans5= "no"

### ANSWER TEST
print(ans5)

[Back to top](#-Index)

### Problem 6

### Confusion Matrix and ROC Curve

**10 Points**

Examine both the confusion matrix and ROC curve using the cell below.  

Create a 1 row by 2 column subplot object and place the confusion matrix on the left and ROC curve on the right.



Use these to determine the number of false positives and false negatives on the test data.  Assign as an integer to `fp` and `fn` below.  Also, use the `RocCurveDisplay` legend to determine the AUC score.  Assign this as a float with two decimal places to `auc` below.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
confusion_matrix(y_test, y_pred)

In [ ]:
### GRADED
fp = ''
fn = ''
auc = ''
# YOUR CODE HERE
cm = ConfusionMatrixDisplay.from_estimator(lgr_pipe, X_test, y_test)
roc_curve_display = RocCurveDisplay.from_estimator(lgr_pipe, X_test, y_test)

fp = 125
fn = 194
auc= 0.86

### ANSWER CHECK
fp, fn, auc


In [ ]:
#roc_curve(y_test, lgr_pipe.predict_proba(X_test)[:,1])
print(y_test.value_counts())
print(lgr_pipe.predict_proba(X_test)[:,1])


[Back to top](#-Index)

### Problem 7

#### What Really Matters

**10 Points**

You see above that you should have 194 False Negatives and 126 False Positives.  Suppose you want to implement an intervention to attempt turning over customers.  To use your classifier, this means being sure about targeting the customers you expect to churn -- in other words minimize the False Negatives.  Use the `predict_proba` method to select the probabilities of the `No` class.  Assign this as an array to `no_probs` below.

In [ ]:
print(lgr_pipe.named_steps['lgr'].classes_)

In [ ]:
### GRADED
no_probs = ''
# YOUR CODE HERE
no_probs = lgr_pipe.predict_proba(X_test)[:,0]

### ANSWER CHECK
no_probs[:5]

[Back to top](#-Index)

### Problem 8

#### Only target customers with high probability

**10 Points**

Even though our classifier is doing better than the baseline, it is still making a high number of mistakes.  Let's only look at the labels for `No` where you are better than 80% sure they are `No`'s.  Select these from your `no_probs` and assign as an array to `high_prob_no` below.

In [ ]:
### GRADED
high_prob_no = ''
# YOUR CODE HERE
high_prob_no = no_probs[no_probs>0.8]

### ANSWER CHECK
high_prob_no[:5]


[Back to top](#-Index)

### Problem 9

#### Number of Predictions

**10 Points**

How many datapoints had probability greater than 80% of `No`?  What percent of the test data is this?  What percent of the original test data set `No` values is this?  Assign your answer as a float to `percent_of_test_data` and `percent_of_no` below.


In [ ]:
### GRADED
percent_of_test_data = ''
percent_of_no = ''
# YOUR CODE HERE
num_dp_prb_no= len(high_prob_no)
percent_of_test_data =  num_dp_prb_no/len(y_test)
percent_of_no = num_dp_prb_no/ y_test.value_counts()["No"]

### ANSWER CHECK
print(percent_of_test_data)
print(percent_of_no)



[Back to top](#-Index)

### Problem 10

#### Important Features

**10 Points**

Now, let us explore the coefficients of the model.  Because the data were scaled, we can think about the coefficients as speaking to a relative feature importance.  Extract the coefficients from your model and sort their absolute values from greatest to least.  Create a DataFrame called `coef_df` that contains the feature name and coefficient. The results begin as shown below:

<table border="1" class="dataframe">  <thead>    <tr style="text-align: right;">      <th></th>      <th>feature</th>      <th>coefs</th>    </tr>  </thead>  <tbody>    <tr>      <th>14</th>      <td>Contract_Two year</td>      <td>1.321160</td>    </tr>    <tr>      <th>20</th>      <td>tenure</td>      <td>1.301754</td>    </tr>    <tr>      <th>9</th>      <td>TechSupport_No internet service</td>      <td>0.753071</td>    </tr>    <tr>      <th>13</th>      <td>Contract_One year</td>      <td>0.701108</td>    </tr>    <tr>      <th>5</th>      <td>InternetService_Fiber optic</td>      <td>0.679121</td>    </tr>  </tbody></table>

In [ ]:
### GRADED
coef_df = ''
# YOUR CODE HERE
# Data was scaled hence the coefs would be interpretable as relative importance of features amongst themselves.

# 1. Get feature names after ColumnTransformer
feature_names_transformed = lgr_pipe.named_steps['transformer'].get_feature_names_out()

# 2. Get the boolean mask from SelectFromModel
selected_features_mask = lgr_pipe.named_steps['selector'].get_support()

# 3. Apply the mask to get the names of selected features
final_feature_names = feature_names_transformed[selected_features_mask]

# 4. Get the coefficients from the LogisticRegression model
coefficients = lgr_pipe.named_steps['lgr'].coef_[0]

# 5. Create a DataFrame
coef_df = pd.DataFrame({
    'feature': final_feature_names,
    'coefs': coefficients
})

# 6. Sort by absolute value of coefficients from greatest to least
coef_df['abs_coefs'] = abs(coef_df['coefs'])
coef_df = coef_df.sort_values(by='abs_coefs', ascending=False).drop(columns=['abs_coefs'])

### ANSWER CHECK
print(coef_df.head())

# print(lgr_pipe.named_steps['lgr'].coef_)
# print(X_train.columns )
# lgr_pipe.named_steps['selector'].get_feature_names_out()

In [ ]:
selected_features_mask

Notice that you should have a higher percentage of No values in your predictions than how much of the data it is comprised of.  In other words, if you randomly selected 50% of the data, you would expect 50% of the No.  Here, by ranking our predictions by probabilities and only selecting those with higher probability we are able to identify almost 70% of the No.  This notion of *LIFT* is an alternative method to that of ROC for understanding the quality of predictions, particularly if you have finite resources to expend.  If you are interested read more [here](https://www.ibm.com/docs/en/spss-statistics/24.0.0?topic=overtraining-cumulative-gains-lift-charts) and the `skplot` library has a straightforward visualization of the lift curve [here](https://scikit-plot.readthedocs.io/en/stable/metrics.html#scikitplot.metrics.plot_cumulative_gain).